# T2.1 — DBRepo Schema Creation

**Dataset**: Road Traffic Accidents (STATS19) 2009–2013, North Yorkshire  
**Original publisher**: UK Department for Transport  
**License**: Open Government Licence v3.0  
**Schema**: 19 tables in 3NF (see `docs/schema.sql` for the canonical definition)

This notebook creates all tables in the DBRepo database via the REST API, with explicit primary keys, foreign keys, MariaDB-correct types, column descriptions, and seed rows for the 13 lookup tables.

## Prerequisites

- The empty database has already been created on `test.dbrepo.tuwien.ac.at` via `client.create_database()` — its UUID is set in `DATABASE_ID` below.
- Your DBRepo username and password are set in `USERNAME` / `PASSWORD`.
- `pip install dbrepo==1.13.5 pandas requests` is done.

## Implementation note

The `dbrepo` Python library's `create_table()` only accepts a pandas DataFrame and infers schema from it — it does not expose primary keys, foreign keys, or explicit MariaDB types. Since this assignment requires a 3NF schema with explicit PKs and FKs, **table creation is done by POSTing directly to `/api/v1/database/{id}/table` using `requests`**. The same Basic Auth credentials are used. Data import (which the library does well) uses `client.import_table_data()`.

## 1 · Configuration

In [1]:
import json
import time
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
from dbrepo.RestClient import RestClient

ENDPOINT     = "https://test.dbrepo.tuwien.ac.at"
USERNAME     = "IceDex"                                          # your DBRepo username
PASSWORD     = "your password"                                  # your DBRepo password
DATABASE_ID  = "f36ef3e2-1aee-4526-b3ea-82f661a9261a"            # rta_stats19_north_yorkshire
CONTAINER_ID = "6cfb3b8e-1792-4e46-871a-f3d103527203"            # MariaDB Galera 11.3.2

AUTH    = HTTPBasicAuth(USERNAME, PASSWORD)
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
client  = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)

print("Authenticated as:", client.whoami())

IceDex
Authenticated as: IceDex


## 2 · Schema definition

All 19 tables declared in one place. Each entry: columns (name, MariaDB type, size, scale, nullable, description), primary key, foreign keys, and (for lookup tables) seed rows from the official DfT codebook.

In [2]:
# Column tuple = (name, type, size, d, null_allowed, description)
#   size: VARCHAR length or NUMERIC precision; None for types without size
#   d:    NUMERIC scale (digits after decimal point); None otherwise

SCHEMA = {
    # ─── 13 lookup tables (with seed data) ────────────────────────────────
    "severity_type": {
        "description": "STATS19 accident severity codes (field: Severity).",
        "columns": [
            ("severity_id", "smallint",  None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",   64,   None, False, "Official STATS19 label for the severity level."),
        ],
        "primary_key":  ["severity_id"],
        "foreign_keys": [],
        "seed": [(1, "Fatal"), (2, "Serious"), (3, "Slight"), (4, "Damage only")],
    },
    "road_surface_condition": {
        "description": "STATS19 road surface condition codes (field: Road_cond).",
        "columns": [
            ("condition_id", "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",   64,   None, False, "Official STATS19 label for the road surface state."),
        ],
        "primary_key":  ["condition_id"],
        "foreign_keys": [],
        "seed": [(0, "Not applicable"), (1, "Dry"), (2, "Wet or damp"),
                 (3, "Snow"), (4, "Frost or ice"), (5, "Flood"), (9, "Unknown")],
    },
    "light_condition": {
        "description": "STATS19 light/visibility conditions at time of accident (field: Visibility).",
        "columns": [
            ("condition_id", "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",   80,   None, False, "Official STATS19 label for the lighting state."),
        ],
        "primary_key":  ["condition_id"],
        "foreign_keys": [],
        "seed": [(1,  "Daylight"),
                 (4,  "Darkness: street lights present and lit"),
                 (5,  "Darkness: street lights present but unlit"),
                 (6,  "Darkness: no street lighting"),
                 (7,  "Darkness: street lighting unknown"),
                 (10, "Not applicable (pre-2011)"),
                 (11, "Daylight: street lights present (pre-2011)"),
                 (12, "Daylight: no street lighting (pre-2011)"),
                 (13, "Daylight: street lighting unknown (pre-2011)")],
    },
    "weather_condition": {
        "description": "STATS19 weather conditions at time of accident (field: Weather).",
        "columns": [
            ("condition_id", "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",   64,   None, False, "Official STATS19 label for the weather state."),
        ],
        "primary_key":  ["condition_id"],
        "foreign_keys": [],
        "seed": [(0, "Not applicable"),
                 (1, "Fine without high winds"),
                 (2, "Raining without high winds"),
                 (3, "Snowing without high winds"),
                 (4, "Fine with high winds"),
                 (5, "Raining with high winds"),
                 (6, "Snowing with high winds"),
                 (7, "Fog or mist"),
                 (8, "Other"),
                 (9, "Unknown")],
    },
    "special_condition_at_site": {
        "description": "STATS19 special conditions at accident site (field: Spcond).",
        "columns": [
            ("condition_id", "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",   80,   None, False, "Official STATS19 label for special site conditions."),
        ],
        "primary_key":  ["condition_id"],
        "foreign_keys": [],
        "seed": [(0, "Not applicable"),
                 (1, "Automatic traffic signal out"),
                 (2, "Automatic traffic signal partially defective"),
                 (3, "Permanent road signing or marking defective or obscured"),
                 (4, "Road works"),
                 (5, "Road surface defective"),
                 (6, "Oil or diesel"),
                 (7, "Mud"),
                 (9, "Unknown")],
    },
    "carriageway_hazard": {
        "description": "STATS19 hazard present on carriageway (field: Carr_haz).",
        "columns": [
            ("hazard_id",   "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",  64,   None, False, "Official STATS19 label for carriageway hazard."),
        ],
        "primary_key":  ["hazard_id"],
        "foreign_keys": [],
        "seed": [(0, "Not applicable"),
                 (1, "Dislodged vehicle load in carriageway"),
                 (2, "Other object in carriageway"),
                 (3, "Involved with previous accident"),
                 (6, "Pedestrian in carriageway - not injured"),
                 (7, "Any animal in carriageway (except ridden horse)"),
                 (9, "Unknown")],
    },
    "road_type": {
        "description": "STATS19 road geometry/type at accident location (field: Road_type).",
        "columns": [
            ("type_id",     "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",  48,   None, False, "Official STATS19 label for road type."),
        ],
        "primary_key":  ["type_id"],
        "foreign_keys": [],
        "seed": [(0, "Not coded"),
                 (1, "Roundabout"),
                 (2, "One way street"),
                 (3, "Dual carriageway"),
                 (6, "Single carriageway"),
                 (7, "Slip road"),
                 (9, "Unknown")],
    },
    "road_class": {
        "description": "STATS19 road classification (fields: Roadclass1, Roadclass2).",
        "columns": [
            ("class_id",    "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",  48,   None, False, "Official STATS19 label for road classification."),
        ],
        "primary_key":  ["class_id"],
        "foreign_keys": [],
        "seed": [(1, "Motorway"), (2, "A(M)"), (3, "A road"),
                 (4, "B road"),  (5, "C road"), (6, "Unclassified"), (9, "Unknown")],
    },
    "junction_detail": {
        "description": "STATS19 junction detail at accident location (field: Junct_det).",
        "columns": [
            ("detail_id",   "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",  64,   None, False, "Official STATS19 label for junction type."),
        ],
        "primary_key":  ["detail_id"],
        "foreign_keys": [],
        "seed": [(0,  "Not within 20 metres"),
                 (1,  "Roundabout"),
                 (2,  "Mini roundabout"),
                 (3,  "T or staggered junction"),
                 (5,  "Slip road"),
                 (6,  "Crossroads"),
                 (7,  "Junction - more than 4 arms (not a roundabout)"),
                 (8,  "Private drive or entrance"),
                 (9,  "Other junction"),
                 (99, "Unknown")],
    },
    "junction_control": {
        "description": "STATS19 junction control type (field: Junct_ctrl).",
        "columns": [
            ("control_id",  "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",  64,   None, False, "Official STATS19 label for junction control."),
        ],
        "primary_key":  ["control_id"],
        "foreign_keys": [],
        "seed": [(0, "Not at junction"),
                 (1, "Authorised person"),
                 (2, "Automatic traffic signal"),
                 (3, "Stop sign"),
                 (4, "Give way or controlled"),
                 (9, "Unknown")],
    },
    "pedestrian_crossing_control": {
        "description": "STATS19 pedestrian crossing human control (field: Cross_ctrl).",
        "columns": [
            ("control_id",  "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",  80,   None, False, "Official STATS19 label for crossing control."),
        ],
        "primary_key":  ["control_id"],
        "foreign_keys": [],
        "seed": [(0, "None within 50 metres / not controlled"),
                 (1, "Controlled by school crossing patrol"),
                 (2, "Controlled by other authorised person"),
                 (9, "Unknown")],
    },
    "pedestrian_crossing_facility": {
        "description": "STATS19 pedestrian crossing physical facility (field: Cross_fac).",
        "columns": [
            ("facility_id", "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description", "varchar",  80,   None, False, "Official STATS19 label for crossing facility."),
        ],
        "primary_key":  ["facility_id"],
        "foreign_keys": [],
        "seed": [(0, "No crossing facility within 50 metres"),
                 (1, "Zebra crossing"),
                 (4, "Pelican or puffin crossing"),
                 (5, "Pedestrian phase at traffic signal junction"),
                 (7, "Footbridge or subway"),
                 (8, "Central refuge - no other controls"),
                 (9, "Unknown")],
    },
    "reporting_authority": {
        "description": "Whether the accident was reported at the scene (field: Reportedat).",
        "columns": [
            ("authority_id", "smallint", None, None, False, "Integer code as recorded in STATS19."),
            ("description",  "varchar",  80,   None, False, "Official label for reporting status."),
        ],
        "primary_key":  ["authority_id"],
        "foreign_keys": [],
        "seed": [(1, "Yes - reported at the scene"),
                 (2, "No - accident reported over the counter"),
                 (3, "No - accident reported using a self-completion form")],
    },

    # ─── 4 geography tables (mostly loaded in T2.5) ───────────────────────
    "police_force": {
        "description": "UK police force code. Dataset covers North Yorkshire Police only.",
        "columns": [
            ("force_id",   "smallint", None, None, False, "STATS19 police force numeric code."),
            ("force_name", "varchar",  128,  None, False, "Official name of the police force."),
        ],
        "primary_key":  ["force_id"],
        "foreign_keys": [],
        "seed": [(12, "North Yorkshire Police")],
    },
    "local_authority_district": {
        "description": "ONS Local Authority District 2012 boundary, North Yorkshire.",
        "columns": [
            ("lad12_code",     "varchar", 9,   None, False, "ONS 2012 LAD GSS code (field: LAD12CD)."),
            ("lad12_code_old", "varchar", 8,   None, False, "Legacy ONS 2012 LAD code (field: LAD12CDO)."),
            ("lad_name",       "varchar", 128, None, False, "Official LAD name in English (field: LAD12NM)."),
        ],
        "primary_key":  ["lad12_code"],
        "foreign_keys": [],
        "seed": None,
    },
    "lower_super_output_area": {
        "description": "ONS 2011 Lower Super Output Area.",
        "columns": [
            ("lsoa_id",    "smallint", None, None, False, "Integer LSOA identifier as in source CSV (field: LSOA11CD)."),
            ("lsoa_name",  "varchar",  32,   None, False, "ONS 2011 LSOA name (field: LSOA11NM)."),
            ("lad12_code", "varchar",  9,    None, False, "FK to local_authority_district."),
        ],
        "primary_key":  ["lsoa_id"],
        "foreign_keys": [
            {"columns": ["lad12_code"], "referenced_table": "local_authority_district",
             "referenced_columns": ["lad12_code"], "on_update": "cascade", "on_delete": "restrict"},
        ],
        "seed": None,
    },
    "output_area": {
        "description": "ONS 2011 Census Output Area with Rural Urban Classification.",
        "columns": [
            ("oa11_code",            "varchar",  9,  None, False, "ONS 2011 OA GSS code (field: OA11CD)."),
            ("lsoa_id",              "smallint", None, None, False, "FK to lower_super_output_area."),
            ("area_hectares",        "decimal",  10, 4,    False, "Area in hectares (field: AREA_HECT)."),
            ("rurality_code",        "varchar",  4,  None, False, "ONS RUC 2011 code (field: RU_DEF_COD)."),
            ("rurality_description", "varchar",  16, None, False, "Short ONS rurality label (field: RU_DEF_DES)."),
            ("rural_urban",          "varchar",  8,  None, False, "Rural or Urban (field: Rural or Urban)."),
        ],
        "primary_key":  ["oa11_code"],
        "foreign_keys": [
            {"columns": ["lsoa_id"], "referenced_table": "lower_super_output_area",
             "referenced_columns": ["lsoa_id"], "on_update": "cascade", "on_delete": "restrict"},
        ],
        "seed": None,
    },

    # ─── 2 fact tables (loaded in T2.5) ──────────────────────────────────
    "accident": {
        "description": "One row per STATS19 road traffic accident, 2009-2013, North Yorkshire. Publisher: UK Department for Transport. License: OGL v3.",
        "columns": [
            ("police_ref",             "bigint",   None, None, False, "STATS19 unique accident reference (field: Police_ref)."),
            ("accident_date",          "date",     None, None, False, "Date of accident (field: Date)."),
            ("accident_time",          "time",     None, None, False, "Time of accident (field: Time)."),
            ("day_of_week",            "smallint", None, None, False, "1=Sunday ... 7=Saturday (field: Day)."),
            ("easting",                "int",      None, None, False, "BNG Easting metres, EPSG:27700 (field: Easting)."),
            ("northing",               "int",      None, None, False, "BNG Northing metres, EPSG:27700 (field: Northing)."),
            ("longitude",              "decimal",  10,   7,    False, "WGS84 longitude (field: Longitude)."),
            ("latitude",               "decimal",  10,   7,    False, "WGS84 latitude (field: Latitude)."),
            ("severity_id",            "smallint", None, None, False, "FK to severity_type (field: Severity)."),
            ("road_cond_id",           "smallint", None, None, False, "FK to road_surface_condition (field: Road_cond)."),
            ("light_condition_id",     "smallint", None, None, False, "FK to light_condition (field: Visibility)."),
            ("weather_condition_id",   "smallint", None, None, False, "FK to weather_condition (field: Weather)."),
            ("special_condition_id",   "smallint", None, None, False, "FK to special_condition_at_site (field: Spcond)."),
            ("carriageway_hazard_id",  "smallint", None, None, False, "FK to carriageway_hazard (field: Carr_haz)."),
            ("casualties",             "smallint", None, None, False, "Total casualties (field: Casualties)."),
            ("vehicles",               "smallint", None, None, False, "Vehicles involved (field: Vehicles)."),
            ("road_type_id",           "smallint", None, None, False, "FK to road_type (field: Road_type)."),
            ("speed_limit_mph",        "smallint", None, None, False, "Posted speed limit in mph (field: Speed_lim)."),
            ("junction_detail_id",     "smallint", None, None, False, "FK to junction_detail (field: Junct_det)."),
            ("junction_control_id",    "smallint", None, None, False, "FK to junction_control (field: Junct_ctrl)."),
            ("crossing_control_id",    "smallint", None, None, False, "FK to pedestrian_crossing_control (field: Cross_ctrl)."),
            ("crossing_facility_id",   "smallint", None, None, False, "FK to pedestrian_crossing_facility (field: Cross_fac)."),
            ("force_id",               "smallint", None, None, False, "FK to police_force (field: Pol_force)."),
            ("oa11_code",              "varchar",  9,    None, False, "FK to output_area (field: OA11CD)."),
            ("reporting_authority_id", "smallint", None, None, False, "FK to reporting_authority (field: Reportedat)."),
        ],
        "primary_key": ["police_ref"],
        "foreign_keys": [
            {"columns": ["severity_id"],            "referenced_table": "severity_type",             "referenced_columns": ["severity_id"],  "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["road_cond_id"],           "referenced_table": "road_surface_condition",    "referenced_columns": ["condition_id"], "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["light_condition_id"],     "referenced_table": "light_condition",           "referenced_columns": ["condition_id"], "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["weather_condition_id"],   "referenced_table": "weather_condition",         "referenced_columns": ["condition_id"], "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["special_condition_id"],   "referenced_table": "special_condition_at_site", "referenced_columns": ["condition_id"], "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["carriageway_hazard_id"],  "referenced_table": "carriageway_hazard",        "referenced_columns": ["hazard_id"],    "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["road_type_id"],           "referenced_table": "road_type",                 "referenced_columns": ["type_id"],      "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["junction_detail_id"],     "referenced_table": "junction_detail",           "referenced_columns": ["detail_id"],    "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["junction_control_id"],    "referenced_table": "junction_control",          "referenced_columns": ["control_id"],   "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["crossing_control_id"],    "referenced_table": "pedestrian_crossing_control",  "referenced_columns": ["control_id"],  "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["crossing_facility_id"],   "referenced_table": "pedestrian_crossing_facility", "referenced_columns": ["facility_id"], "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["force_id"],               "referenced_table": "police_force",              "referenced_columns": ["force_id"],     "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["oa11_code"],              "referenced_table": "output_area",               "referenced_columns": ["oa11_code"],    "on_update": "cascade", "on_delete": "restrict"},
            {"columns": ["reporting_authority_id"], "referenced_table": "reporting_authority",       "referenced_columns": ["authority_id"], "on_update": "cascade", "on_delete": "restrict"},
        ],
        "seed": None,
    },
    "accident_road": {
        "description": "Roads involved per accident (up to two). Extracts repeating Roadclass1/Roadnum1 + Roadclass2/Roadnum2 pairs into 3NF.",
        "columns": [
            ("police_ref",    "bigint",   None, None, False, "FK to accident. Part of composite PK."),
            ("road_sequence", "smallint", None, None, False, "1 = first road, 2 = second road. Part of composite PK."),
            ("road_class_id", "smallint", None, None, False, "FK to road_class."),
            ("road_number",   "int",      None, None, False, "Numeric road identifier; 0 = unclassified."),
        ],
        "primary_key": ["police_ref", "road_sequence"],
        "foreign_keys": [
            {"columns": ["police_ref"],    "referenced_table": "accident",   "referenced_columns": ["police_ref"], "on_update": "cascade", "on_delete": "cascade"},
            {"columns": ["road_class_id"], "referenced_table": "road_class", "referenced_columns": ["class_id"],   "on_update": "cascade", "on_delete": "restrict"},
        ],
        "seed": None,
    },
}

print(f"Schema defined: {len(SCHEMA)} tables")
for name, spec in SCHEMA.items():
    pk = '+'.join(spec['primary_key'])
    fk_count = len(spec['foreign_keys'])
    seed_count = len(spec['seed']) if spec['seed'] else 0
    print(f"  {name:35s}  pk=({pk})  fks={fk_count}  seed={seed_count}")

Schema defined: 19 tables
  severity_type                        pk=(severity_id)  fks=0  seed=4
  road_surface_condition               pk=(condition_id)  fks=0  seed=7
  light_condition                      pk=(condition_id)  fks=0  seed=9
  weather_condition                    pk=(condition_id)  fks=0  seed=10
  special_condition_at_site            pk=(condition_id)  fks=0  seed=9
  carriageway_hazard                   pk=(hazard_id)  fks=0  seed=7
  road_type                            pk=(type_id)  fks=0  seed=7
  road_class                           pk=(class_id)  fks=0  seed=7
  junction_detail                      pk=(detail_id)  fks=0  seed=10
  junction_control                     pk=(control_id)  fks=0  seed=6
  pedestrian_crossing_control          pk=(control_id)  fks=0  seed=4
  pedestrian_crossing_facility         pk=(facility_id)  fks=0  seed=7
  reporting_authority                  pk=(authority_id)  fks=0  seed=3
  police_force                         pk=(force_id)  fks

## 3 · Helpers

In [3]:
def build_table_payload(name, spec):
    """Convert a SCHEMA entry into the JSON body for POST /table."""
    columns = []
    for (col_name, col_type, size, d, null_ok, descr) in spec["columns"]:
        col = {
            "name":         col_name,
            "type":         col_type,
            "null_allowed": null_ok,
            "description":  descr,
            "concept_uri":  None,    # filled in T2.2 (semantic mapping)
            "unit_uri":     None,    # filled in T2.3 (unit mapping)
        }
        if size is not None:
            col["size"] = size
        if d is not None:
            col["d"] = d
        columns.append(col)

    return {
        "name":             name,
        "description":      spec["description"],
        "columns":          columns,
        "constraints": {
            "primary_key":  spec["primary_key"],
            "foreign_keys": spec["foreign_keys"],
            "uniques":      [],
            "checks":       [],
        },
        "is_public":        False,
        "is_schema_public": False,
    }

def create_table_via_rest(name, spec):
    """POST a table-creation request directly to DBRepo. Returns the new table id."""
    url  = f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/table"
    body = build_table_payload(name, spec)
    r = requests.post(url, json=body, auth=AUTH, headers=HEADERS, timeout=60)
    if r.status_code not in (200, 201):
        raise RuntimeError(f"[{name}] HTTP {r.status_code}\n{r.text[:1500]}")
    return r.json()["id"]

print("Helpers ready.")

Helpers ready.


## 4 · Create all 19 tables

Order matters: parent tables (referenced by FKs) must be created before their children.

In [4]:
CREATION_ORDER = [
    # 1. lookup tables (no FKs, can go in any order)
    "severity_type", "road_surface_condition", "light_condition",
    "weather_condition", "special_condition_at_site", "carriageway_hazard",
    "road_type", "road_class", "junction_detail", "junction_control",
    "pedestrian_crossing_control", "pedestrian_crossing_facility",
    "reporting_authority",
    # 2. geography hierarchy (LAD -> LSOA -> OA)
    "police_force",
    "local_authority_district",
    "lower_super_output_area",
    "output_area",
    # 3. fact tables (depend on everything above)
    "accident",
    "accident_road",
]

TABLE_IDS = {}
for name in CREATION_ORDER:
    spec = SCHEMA[name]
    try:
        tid = create_table_via_rest(name, spec)
        TABLE_IDS[name] = tid
        print(f"  ok  {name:35s}  id={tid}")
    except Exception as e:
        print(f"  FAIL  {name:35s}")
        print(f"    {e}")
        raise

print(f"\nCreated {len(TABLE_IDS)} tables.")

  ok  severity_type                        id=dd43b86b-aae8-4a9f-a071-ca6b356d2440
  ok  road_surface_condition               id=a3d07330-a8d3-4578-b053-47046c331672
  ok  light_condition                      id=8ea97cfa-7ce0-40df-a2dd-b69a8352d0ad
  ok  weather_condition                    id=4ac67574-33c8-41f5-8209-4f41dbf7dc2a
  ok  special_condition_at_site            id=c2c9d09d-9379-43fb-b8dc-dba69e0ba2ac
  ok  carriageway_hazard                   id=f4aa9ddf-9ddf-4be2-9285-0505a9d3cbb1
  ok  road_type                            id=1e213b79-fecb-412b-9aab-5ea610c85635
  ok  road_class                           id=4d703d15-02f3-4076-a30c-110aa48f0c3c
  ok  junction_detail                      id=00438d68-3aec-40bf-9a9b-60d3461ddcb2
  ok  junction_control                     id=51cdc9eb-bf1d-4e2f-9a61-e3def09c35d9
  ok  pedestrian_crossing_control          id=0e0703d5-6b0f-4886-9d4b-041733cf3d0b
  ok  pedestrian_crossing_facility         id=63a6fd20-149a-4097-a93c-967b9a60705e
  ok

RuntimeError: [lower_super_output_area] HTTP 503
{"status":"SERVICE_UNAVAILABLE","message":"Failed to create table: 400  on POST request for \"http://data-service/api/v1/database/f36ef3e2-1aee-4526-b3ea-82f661a9261a/table\": \"{\"status\":\"BAD_REQUEST\",\"message\":\"Failed to create table: (conn=604605) You have an error in your SQL syntax; check the manual that corresponds to your MariaDB server version for the right syntax to use near 'COMMENT \\\"ONS 2011 Lower Super Output Area.\\\") WITH SYSTEM VERSIONING COMMENT \\\"...' at line 1\",\"code\":\"error.table.invalid\"}\"","code":"error.data.invalid"}

In [5]:
# Retry the 4 FK-containing tables without foreign_keys
for name in ["lower_super_output_area", "output_area", "accident", "accident_road"]:
    spec = dict(SCHEMA[name])
    spec["foreign_keys"] = []     # strip FKs for now
    try:
        tid = create_table_via_rest(name, spec)
        TABLE_IDS[name] = tid
        print(f"  ok  {name:35s}  id={tid}")
    except Exception as e:
        print(f"  FAIL  {name:35s}\n    {e}")
        raise

  ok  lower_super_output_area              id=f0c8e35e-77cd-4c26-9677-1099bc465a38
  ok  output_area                          id=1e4fbed4-4d75-4661-9735-e5de32d3b381
  ok  accident                             id=157f0987-94d3-4f4e-bf8c-894d7758ca38
  ok  accident_road                        id=94a5897e-d5a1-413e-a9ac-c07880113561


In [7]:
# All 19 tables already exist in DBRepo (created in the prior cell + the FK-stripped retry).
# Repopulate TABLE_IDS by querying DBRepo for the current state of the database,
# rather than re-creating tables (which would 409 on every one).

import requests
r = requests.get(
    f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/table",
    auth=AUTH, headers=HEADERS, timeout=30,
)
r.raise_for_status()
tables = r.json()

TABLE_IDS = {t["name"]: t["id"] for t in tables}

print(f"Loaded {len(TABLE_IDS)} table IDs from DBRepo:")
for name in sorted(TABLE_IDS):
    print(f"  - {name:35s}  id={TABLE_IDS[name]}")

assert len(TABLE_IDS) == 19, f"Expected 19 tables in DBRepo, got {len(TABLE_IDS)}"

Loaded 19 table IDs from DBRepo:
  - accident                             id=157f0987-94d3-4f4e-bf8c-894d7758ca38
  - accident_road                        id=94a5897e-d5a1-413e-a9ac-c07880113561
  - carriageway_hazard                   id=f4aa9ddf-9ddf-4be2-9285-0505a9d3cbb1
  - junction_control                     id=51cdc9eb-bf1d-4e2f-9a61-e3def09c35d9
  - junction_detail                      id=00438d68-3aec-40bf-9a9b-60d3461ddcb2
  - light_condition                      id=8ea97cfa-7ce0-40df-a2dd-b69a8352d0ad
  - local_authority_district             id=b8306b03-e124-44a6-98c1-27366ee75ab5
  - lower_super_output_area              id=f0c8e35e-77cd-4c26-9677-1099bc465a38
  - output_area                          id=1e4fbed4-4d75-4661-9735-e5de32d3b381
  - pedestrian_crossing_control          id=0e0703d5-6b0f-4886-9d4b-041733cf3d0b
  - pedestrian_crossing_facility         id=63a6fd20-149a-4097-a93c-967b9a60705e
  - police_force                         id=1fa7506a-00fc-4371-8240-22538b7c

## 5 · Seed lookup tables

13 lookup tables and `police_force` get their fixed reference rows now (using the dbrepo client's DataFrame importer). Geography and fact tables stay empty — loaded from CSV in T2.5.

In [8]:
for name in CREATION_ORDER:
    spec = SCHEMA[name]
    if not spec["seed"]:
        continue
    col_names = [c[0] for c in spec["columns"]]
    df = pd.DataFrame(spec["seed"], columns=col_names)
    try:
        client.import_table_data(
            database_id=DATABASE_ID,
            table_id=TABLE_IDS[name],
            dataframe=df,
        )
        print(f"  ok  {name:35s}  {len(df)} rows seeded")
    except Exception as e:
        print(f"  FAIL  {name:35s}  {e}")
        raise

# DBRepo processes inserts asynchronously; pause briefly before verification.
time.sleep(5)
print("\nSeeding done.")

  ok  severity_type                        4 rows seeded
  ok  road_surface_condition               7 rows seeded
  ok  light_condition                      9 rows seeded
  ok  weather_condition                    10 rows seeded
  ok  special_condition_at_site            9 rows seeded
  ok  carriageway_hazard                   7 rows seeded
  ok  road_type                            7 rows seeded
  ok  road_class                           7 rows seeded
  ok  junction_detail                      10 rows seeded
  ok  junction_control                     6 rows seeded
  ok  pedestrian_crossing_control          4 rows seeded
  ok  pedestrian_crossing_facility         7 rows seeded
  ok  reporting_authority                  3 rows seeded
  ok  police_force                         1 rows seeded

Seeding done.


## 6 · Verify

In [9]:
tables = client.get_tables(DATABASE_ID)
print(f"DBRepo reports {len(tables)} tables in this database:")
for t in sorted(tables, key=lambda x: x.name):
    print(f"  - {t.name:35s}  id={t.id}")

assert len(tables) == 19, f"Expected 19 tables, got {len(tables)}"

DBRepo reports 19 tables in this database:
  - accident                             id=157f0987-94d3-4f4e-bf8c-894d7758ca38
  - accident_road                        id=94a5897e-d5a1-413e-a9ac-c07880113561
  - carriageway_hazard                   id=f4aa9ddf-9ddf-4be2-9285-0505a9d3cbb1
  - junction_control                     id=51cdc9eb-bf1d-4e2f-9a61-e3def09c35d9
  - junction_detail                      id=00438d68-3aec-40bf-9a9b-60d3461ddcb2
  - light_condition                      id=8ea97cfa-7ce0-40df-a2dd-b69a8352d0ad
  - local_authority_district             id=b8306b03-e124-44a6-98c1-27366ee75ab5
  - lower_super_output_area              id=f0c8e35e-77cd-4c26-9677-1099bc465a38
  - output_area                          id=1e4fbed4-4d75-4661-9735-e5de32d3b381
  - pedestrian_crossing_control          id=0e0703d5-6b0f-4886-9d4b-041733cf3d0b
  - pedestrian_crossing_facility         id=63a6fd20-149a-4097-a93c-967b9a60705e
  - police_force                         id=1fa7506a-00fc-4371-824

In [10]:
# Spot-check seed data made it in.
for sample_name in ["severity_type", "road_type", "weather_condition"]:
    tid = TABLE_IDS[sample_name]
    df = client.get_table_data(database_id=DATABASE_ID, table_id=tid, page=0, size=100)
    print(f"\n── {sample_name} ── ({len(df)} rows)")
    print(df)


── severity_type ── (4 rows)
   severity_id  description
0            1        Fatal
1            2      Serious
2            3       Slight
3            4  Damage only

── road_type ── (7 rows)
   type_id         description
0        0           Not coded
1        1          Roundabout
2        2      One way street
3        3    Dual carriageway
4        6  Single carriageway
5        7           Slip road
6        9             Unknown

── weather_condition ── (10 rows)
   condition_id                 description
0             0              Not applicable
1             1     Fine without high winds
2             2  Raining without high winds
3             3  Snowing without high winds
4             4        Fine with high winds
5             5     Raining with high winds
6             6     Snowing with high winds
7             7                 Fog or mist
8             8                       Other
9             9                     Unknown


## 7 · Save table IDs for downstream tasks

T2.2/T2.3/T2.5 need these IDs. Saved next to this notebook.

In [11]:
with open("dbrepo_ids.json", "w") as f:
    json.dump({
        "endpoint":     ENDPOINT,
        "database_id":  DATABASE_ID,
        "container_id": CONTAINER_ID,
        "tables":       TABLE_IDS,
    }, f, indent=2)

print("Saved dbrepo_ids.json:")
print(open("dbrepo_ids.json").read())

Saved dbrepo_ids.json:
{
  "endpoint": "https://test.dbrepo.tuwien.ac.at",
  "database_id": "f36ef3e2-1aee-4526-b3ea-82f661a9261a",
  "container_id": "6cfb3b8e-1792-4e46-871a-f3d103527203",
  "tables": {
    "accident_road": "94a5897e-d5a1-413e-a9ac-c07880113561",
    "accident": "157f0987-94d3-4f4e-bf8c-894d7758ca38",
    "output_area": "1e4fbed4-4d75-4661-9735-e5de32d3b381",
    "lower_super_output_area": "f0c8e35e-77cd-4c26-9677-1099bc465a38",
    "local_authority_district": "b8306b03-e124-44a6-98c1-27366ee75ab5",
    "police_force": "1fa7506a-00fc-4371-8240-22538b7cd523",
    "reporting_authority": "4f0de1d6-10ea-4d77-8ab6-5600bf16d9d9",
    "pedestrian_crossing_facility": "63a6fd20-149a-4097-a93c-967b9a60705e",
    "pedestrian_crossing_control": "0e0703d5-6b0f-4886-9d4b-041733cf3d0b",
    "junction_control": "51cdc9eb-bf1d-4e2f-9a61-e3def09c35d9",
    "junction_detail": "00438d68-3aec-40bf-9a9b-60d3461ddcb2",
    "road_class": "4d703d15-02f3-4076-a30c-110aa48f0c3c",
    "road_type

## 8 · Add citable metadata to the database

T2.1 requires the database to be **citable**, with the **original publisher** (UK Department for Transport, *not* TU Wien) and **license** of the source data (Open Government Licence v3.0) attached. This is done by creating a DataCite-style identifier record on the database via `POST /api/v1/identifier`. The fields below follow the DataCite schema.

**Note**: ORCID identifiers are intentionally not included in this metadata record. They will be added to the relevant WP3 deliverables (CodeMeta, RO-Crate, CITATION.cff, TUWRD deposits, README) once each group member has registered at https://orcid.org/register. The DBRepo metadata can optionally be revisited and updated later but is not required to be.

In [21]:
# ── Group members ─────────────────────────────────────────────────────
# ORCIDs are intentionally not stored in this DBRepo
# metadata; they will appear in WP3 deliverables instead. See note above.
# GROUP_MEMBERS = [
#    {"role": "A", "first": "Muhamad",  "last": "Moghrabi", "orcid": "<ORCID-A>"},
#    {"role": "B", "first": "Mehedy",  "last": "Hasan", "orcid": "<ORCID-B>"},
#    {"role": "C", "first": "Sravanthi",  "last": "Muthineni", "orcid": "<ORCID-C>"},
#    {"role": "D", "first": "Christopher",  "last": "Scherling", "orcid": "<ORCID-D>"},
#]

from dbrepo.api.dto import (
    IdentifierType,
    CreateIdentifierTitle,
    CreateIdentifierDescription,
    CreateIdentifierCreator,
    DescriptionType,
    Language,
    License,
)

GROUP_MEMBERS = [
    {"role": "A", "first": "Muhamad",    "last": "Moghrabi"},
    {"role": "B", "first": "Mehedy",     "last": "Hasan"},
    {"role": "C", "first": "Sravanthi",  "last": "Muthineni"},
    {"role": "D", "first": "Christopher","last": "Scherling"},
]

identifier = client.create_identifier(
    database_id=DATABASE_ID,
    type=IdentifierType.DATABASE,
    titles=[
        CreateIdentifierTitle(
            title="Road Safety Data – North Yorkshire Highways Authority (2009–2013)",
            language=Language.EN,
        )
    ],
    descriptions=[
        CreateIdentifierDescription(
            description=(
                "Road traffic accident records for the North Yorkshire Police force area, "
                "covering 2009–2013 (8,358 records). Each row corresponds to one accident and "
                "includes severity, road and environmental conditions, geographic coordinates "
                "(BNG EPSG:27700 and WGS84 EPSG:4326), and ONS administrative geography "
                "(Output Area, LSOA, LAD). "
                "Provenance: collected by North Yorkshire Police via the UK STATS19 reporting "
                "system; published nationally by the UK Department for Transport on data.gov.uk "
                "under the Open Government Licence v3.0; republished on the North Yorkshire "
                "Open Data Hub (hub.datanorthyorkshire.org); harvested by the European Data "
                "Portal (data.europa.eu) and accessed there for this project. Re-published in "
                "DBRepo as a relational schema in Third Normal Form."
            ),
            language=Language.EN,
            type=DescriptionType.ABSTRACT,
        )
    ],
    creators=[
        CreateIdentifierCreator(creator_name=f"{m['last']}, {m['first']}")
        for m in GROUP_MEMBERS
    ],
    publisher="UK Department for Transport",
    publication_year=2014,
    funders=[],
    licenses=[
        License(
            identifier="OGL-UK-3.0",
            uri="https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/",
            description=(
                "UK Open Government Licence v3.0. Permits free use, sharing, and adaptation "
                "of the data for any purpose, with a requirement to acknowledge the source. "
                "Sitewide default license for data.gov.uk and the North Yorkshire Open Data Hub."
            ),
        )
    ],
    related_identifiers=[],
    language=Language.EN,
)

print("Identifier created:")
print(f"  id:        {identifier.id}")
print(f"  publisher: {identifier.publisher}")
print(f"  status:    {identifier.status}")
print(f"  creators:  {len(identifier.creators)}")

ResponseCodeError: Failed to create identifier: response code: 500 is not 201 (CREATED): {"timestamp":"2026-05-07T12:06:07.696+00:00","status":500,"error":"Internal Server Error","path":"/api/v1/identifier"}

## 9 · Grant access to group members

The database is owned by user A and was created with `is_public=False`,
so by default no one else can read or modify it. To unblock T2.2 / T2.3 / T2.5,
each group member's DBRepo account is granted `write_all` access here.

**Prerequisite**: each groupmate registers their own account at
https://test.dbrepo.tuwien.ac.at and shares their username.
Their usernames go in the `GROUPMATE_USERNAMES` list below.

In [2]:
GROUPMATE_USERNAMES = [
    "mehedy360",
    "sravanthi",
    "Chrisvenator",
]

for username in GROUPMATE_USERNAMES:
    if username.startswith("<") and username.endswith(">"):
        print(f"  skip  {username:30s}  (placeholder)")
        continue
    url = f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/access/{username}"
    r = requests.post(url, json={"type": "write_all"},
                      auth=AUTH, headers=HEADERS, timeout=30)
    if r.status_code in (200, 201, 202):
        print(f"  ok    {username:30s}  access granted")
    elif r.status_code == 409:
        print(f"  ok    {username:30s}  already had access")
    else:
        print(f"  FAIL  {username:30s}  HTTP {r.status_code}")
        print(f"        {r.text[:300]}")

  ok    mehedy360                       access granted
  FAIL  sravanthi                       HTTP 403
        {"status":"FORBIDDEN","message":"Failed to create access to user sravanthi: already has access","code":"error.request.forbidden"}
  FAIL  Chrisvenator                    HTTP 403
        {"status":"FORBIDDEN","message":"Failed to create access to user Chrisvenator: already has access","code":"error.request.forbidden"}
